In [23]:
#setup

#conda create --name sds-project python=3.12
#conda install --channel conda-forge pandas numpy matplotlib requests geopandas
#conda install --channel conda-forge ipykernel

import pandas as pd
import requests
import geopandas as gpd
import folium
import numpy as np

In [24]:
MAP_KEY = '5eae605403f5deded880b550afef3667'

def get_transaction_count() :
  count = 0
  try:
    response = requests.get(url = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=' + MAP_KEY)
    data = response.json()
    df = pd.Series(data)
    count = df['current_transactions']
  except:
    print ("Error in our call.")
  return count

In [25]:
#sensors:
da_url = 'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/' + MAP_KEY + '/all'
daterange_df = pd.read_csv(da_url)

#ToDo: format as pd.datetime

display(daterange_df)

#set one sensor
sensor = daterange_df["data_id"][2] #VIIRS has better resolution than other sensors (375m vs 1000m), NRT means only a few minutes lag
print("Current sensor name: ", sensor)
daterange_df.info()

,data_id,min_date,max_date
0,MODIS_NRT,2026-02-01,2026-05-09
1,MODIS_SP,2000-11-01,2026-01-31
2,VIIRS_NOAA20_NRT,2026-03-01,2026-05-09
3,VIIRS_NOAA20_SP,2018-04-01,2026-02-28
4,VIIRS_NOAA21_NRT,2024-01-17,2026-05-09
5,VIIRS_SNPP_NRT,2026-03-01,2026-05-09
6,VIIRS_SNPP_SP,2012-01-20,2026-02-28
7,LANDSAT_NRT,2022-06-20,2026-05-08
8,GOES_NRT,2022-08-09,2026-05-09
9,BA_MODIS,2000-11-01,2026-02-01


Current sensor name:  VIIRS_NOAA20_NRT
<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   data_id   11 non-null     str  
 1   min_date  11 non-null     str  
 2   max_date  11 non-null     str  
dtypes: str(3)
memory usage: 396.0 bytes


In [26]:
#retrieve data 

area = "0,35,20,70" #either "world" or bbox lonmin,latmin,lonmax,latmax, e. g. 0,35,20,70
day_range = 1 #in range(1,5)
date = None
area_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + f'/{sensor}/{area}/{day_range}'
start_count = get_transaction_count()
df_area = pd.read_csv(area_url)
end_count = get_transaction_count()
print ('We used %i transactions.' % (end_count-start_count))

df_area

We used 2 transactions.


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,57.06199,9.98077,309.52,0.38,0.43,2026-05-09,105,N20,VIIRS,n,2.0NRT,281.19,1.68,N
1,57.06253,9.97440,314.71,0.38,0.43,2026-05-09,105,N20,VIIRS,n,2.0NRT,280.22,1.68,N
2,58.36064,12.37907,327.92,0.50,0.41,2026-05-09,105,N20,VIIRS,n,2.0NRT,279.28,4.57,N
3,58.68362,17.13652,317.08,0.40,0.37,2026-05-09,105,N20,VIIRS,n,2.0NRT,278.34,1.51,N
4,48.27274,14.34239,297.64,0.38,0.36,2026-05-09,107,N20,VIIRS,n,2.0NRT,282.40,2.39,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
213,51.02924,2.27494,300.19,0.54,0.51,2026-05-09,249,N20,VIIRS,n,2.0NRT,281.73,1.36,N
214,51.03160,2.28501,314.70,0.54,0.51,2026-05-09,249,N20,VIIRS,n,2.0NRT,282.51,4.35,N
215,51.03354,2.27789,338.74,0.54,0.51,2026-05-09,249,N20,VIIRS,n,2.0NRT,281.67,4.35,N
216,51.03591,2.28797,316.12,0.54,0.51,2026-05-09,249,N20,VIIRS,n,2.0NRT,282.59,4.35,N


In [27]:
#clean df_area:

#remove low confidence entries
mask = df_area['confidence'] != "l"
print((df_area['confidence'] == "l").sum(), "Entries removed due to low confidence")
df_area = df_area[mask]

#format datetime
df_area["acq_datetime"] = pd.to_datetime(df_area['acq_date'] + df_area['acq_time'].astype(str).str.zfill(4),  # Zero-pad to 4 digits (e.g., '626' -> '0626')
    format='%Y-%m-%d%H%M', errors='coerce'
)

df_area = df_area.drop(columns = ["acq_date", "acq_time"])

0 Entries removed due to low confidence


In [28]:

area_gpd = gpd.GeoDataFrame(
    df_area, geometry=gpd.points_from_xy(df_area["longitude"], df_area["latitude"], crs = 4326)
)

vmin = df_area['frp'].quantile(0.02)
vmax = df_area['frp'].quantile(0.98)

#area_gpd.explore(column = "frp", cmap = "YlOrRd", vmin = vmin, vmax = vmax)

area_gpd_sub = area_gpd[['latitude', 'longitude', 'frp', 'acq_datetime', 'geometry']]
area_gpd_sub['acq_datetime'] = area_gpd_sub['acq_datetime'].astype(str)



In [29]:
# Initialize the basemap centered on Zurich
m4 = folium.Map(tiles="CartoDB Positron")

# Add the layer with a custom marker parameter
#marker from the font awesom library ('fa')
folium.GeoJson(
    area_gpd_sub,
    name="Wildfires",
    tooltip=folium.GeoJsonTooltip(fields=["frp"], aliases=["Fire Reactive Power:"]),
    # Override the default teardrop with a blue bicycle icon
    marker=folium.Marker(  # <- THIS IS NEW
        icon=folium.Icon(color="red", icon="fire", prefix="fa", icon_color= "orange")
    ),
).add_to(m4)

# Add the interactive layer control menu
folium.LayerControl().add_to(m4)
m4

In [30]:
from folium.plugins import MarkerCluster

# Initialize the basemap
m6 = folium.Map(tiles="CartoDB Positron")

#add hover over single icons
folium.GeoJson(
    area_gpd_sub,
    name="Individual wildfires",
    show = False, #default off
    tooltip=folium.GeoJsonTooltip(fields=["frp"], aliases=["Fire Reactive Power:"]),
    # Override the default teardrop with a blue bicycle icon
    marker=folium.Marker(  # <- THIS IS NEW
        icon=folium.Icon(color="red", icon="fire", prefix="fa", icon_color= "orange")
    ),
).add_to(m6)




marker_cluster = MarkerCluster(name="Scaled Wildfire Clusters", cmap = "YlOrRd").add_to(m6)

for idx, row in area_gpd_sub.iterrows():
    lat = row.geometry.y
    lon = row.geometry.x
    frp = row["frp"]
    tooltip_text = f"Fire Reactive Power: {frp}"

    # Mathematical Scaling Logic:
    # Base size of 14px, plus an increase based on the square root of the frp
    icon_size = 14 + (np.sqrt(frp) * 25)  # <- THIS IS NEW

    # Injecting custom CSS to draw a perfect circle with our dynamic size (NEW)
    icon_html = f"""
        <div style="
            font-size: {icon_size}px;
            color: #FF4500;
            background: #FF8C00;
            border-radius: 50%;
            width: {icon_size}px;
            height: {icon_size}px;
            display: flex;
            align-items: center;
            justify-content: center;
            border: 1px solid #1f77b4;">
            <i class="fa fa-fire"></i>
        </div>"""

    # Apply the custom HTML using DivIcon
    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(  # <- THIS IS NEW
            html=icon_html,
            icon_size=(icon_size, icon_size),
            icon_anchor=(icon_size / 2, icon_size / 2),  # Centers the icon perfectly
        ),
        tooltip=tooltip_text,
    ).add_to(marker_cluster)


# Add the interactive layer control menu
folium.LayerControl().add_to(m6)

m6